# ATML PA3 — Task 1 training

**Kaggle:** Accelerator = *GPU T4 x2*, Internet = on. Add the `atml-pa3-code` dataset.
Then **Save Version → Save & Run All (Commit)** so it runs in the background (no disconnects).
Each GPU trains one variant. Suggested sessions (keeps the 20 GB output disk safe):

| Session | GPU 0 | GPU 1 |
|---|---|---|
| 1 | `full` | `lora4` |
| 2 | `l2sp` | `lora16` |
| 3 | `oplora16` | (eval of finished runs) |

**Resuming:** if a run was cut off, add the previous version's output as an input (Add Input →
Your Work → this notebook) — the cell below copies any `resume/` state back, and training continues.

**Colab:** runs one variant on the single T4; outputs go to Drive `MyDrive/atml-pa3/outputs`.
Set `EXTRA_SAVE` to e.g. 100 so a disconnect loses little work, then just rerun all cells.

In [ ]:
# --- locate code (Kaggle dataset "atml-pa3-code" containing atml-pa3-code.zip, or Colab Drive) ---
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content/drive/MyDrive/atml-pa3"
if not ON_KAGGLE:
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
REPO = "/tmp/ATML-PA3"           # code lives outside the saved output
zips = glob.glob("/kaggle/input/**/atml-pa3-code.zip", recursive=True) + glob.glob(f"{WORK}/atml-pa3-code.zip")
# Kaggle usually unpacks an uploaded zip, so also accept an already-extracted copy
dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/task1/train.py", recursive=True)]
shutil.rmtree(REPO, ignore_errors=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(REPO)
else:
    assert dirs, "upload outputs/atml-pa3-code.zip as a Kaggle dataset (or to Drive/atml-pa3/)"
    shutil.copytree(dirs[0], REPO)
print("code from", zips[:1] or dirs[:1])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "hf_xet"], check=True)
os.chdir(REPO)
os.makedirs(f"{WORK}/outputs", exist_ok=True)
if not os.path.exists("outputs"):
    os.symlink(f"{WORK}/outputs", "outputs")
import torch; print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
VARIANTS = ["l2sp", "lora16"]     # <- edit per session (one per GPU); session 3: ["oplora16"]
EXTRA_SAVE = 0 if ON_KAGGLE else 100
TIME_LIMIT_H = 11.3 if ON_KAGGLE else 11.0   # save + exit cleanly before the platform limit

In [ ]:
# restore earlier runs (previous notebook versions attached as inputs)
for prev in glob.glob("/kaggle/input/*/outputs/task1/*") if ON_KAGGLE else []:
    v = os.path.basename(prev)
    dst = f"outputs/task1/{v}"
    if not os.path.exists(dst):
        print("restoring", prev, "->", dst)
        shutil.copytree(prev, dst)

In [ ]:
import time
n_gpu = max(torch.cuda.device_count(), 1)
procs = {}
def launch(v, gpu):
    os.makedirs(f"outputs/task1/{v}", exist_ok=True)
    log = open(f"outputs/task1/{v}/stdout.log", "a")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
               TOKENIZERS_PARALLELISM="false")
    cmd = [sys.executable, "-m", "task1.train", "--variant", v, "--out", f"outputs/task1/{v}",
           "--extra_save_every", str(EXTRA_SAVE), "--time_limit_h", str(TIME_LIMIT_H)]
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)

queue = list(VARIANTS)
free = list(range(n_gpu))
while queue or procs:
    while queue and free:
        g = free.pop(0); v = queue.pop(0); procs[v] = (launch(v, g), g); print("started", v, "on gpu", g)
    time.sleep(120)
    for v, (p, g) in list(procs.items()):
        tail = open(f"outputs/task1/{v}/stdout.log").read().splitlines()[-1:]
        print(time.strftime("%H:%M"), v, tail, flush=True)
        if p.poll() is not None:
            print(v, "exited with", p.returncode); procs.pop(v); free.append(g)

In [ ]:
for v in VARIANTS:
    print(v, open(f"outputs/task1/{v}/summary.json").read() if os.path.exists(f"outputs/task1/{v}/summary.json") else "no summary")
!du -sh outputs/task1/*